In [16]:
import os
import urllib.parse
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.agent_toolkits import create_sql_agent


In [17]:
# =====================================================================
# DATABASE CONNECTION (CRITICAL SECURITY STEP: USE A READ-ONLY USER)
# =====================================================================
load_dotenv()  # Load environment variables from .env file

db_user = "root"
raw_password = os.getenv("DB_PASSWORD") 
db_host = "localhost"
db_name = "atliq_tshirts"

# 1. Safely escape special characters in the password
# This turns "@" into "%40", ":" into "%3A", etc.
safe_password = urllib.parse.quote_plus(raw_password)

# 2. Build the URI using the sanitized password
db_uri = f"mysql+pymysql://{db_user}:{safe_password}@{db_host}/{db_name}"

# 3. Connect safely without parsing issues
db = SQLDatabase.from_uri(db_uri, sample_rows_in_table_info=3)

print(db.table_info)


CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci

/*
3 rows from discounts table:
discount_id	t_shirt_id	pct_discount
1	1	10.00
2	2	15.00
3	3	20.00
*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci

/*
3 rows from t_shirts table:
t_shirt_id	brand	color	size	price	stock

In [18]:


llm = ChatGroq(
    model="llama-3.3-70b-versatile", # High-reasoning 70B model
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0 # CRITICAL: keep at 0 for deterministic SQL generation
)

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()



In [ ]:
agent_executor = create_sql_agent(llm, toolkit=toolkit, agent_type="tool-calling", verbose=True)

safe_prompt = "How many t-shirts do we have left for nike in extra small size and white color?"
response = agent_executor.invoke({"input": safe_prompt})
print(response)